# Building Multimodal RAG Application #5: Vector Retrieval

This Colab-ready companion notebook creates paired image–caption embeddings, stores them in LanceDB, enriches short transcripts with neighboring context, and retrieves ranked visual evidence for a text query. It uses the public CLIP checkpoint directly so no hosted embedding API key is required.

In [ ]:
!pip -q install lancedb transformers torch pillow pandas numpy

In [ ]:
from pathlib import Path
import json

import lancedb
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import torch
from transformers import AutoProcessor, CLIPModel

## Create a tiny stand-alone video-frame corpus

For a real project, replace these generated frames with the `frame_metadata.json` produced by Part 4. Each record must keep its source video, timestamp, frame path, and transcript or visual caption.

In [ ]:
DATA_DIR = Path("multimodal_rag_retrieval")
FRAMES_DIR = DATA_DIR / "frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

examples = [
    ("space", "An astronaut works outside a spacecraft above Earth.", "navy"),
    ("playground", "A toddler plays on a colorful slide with an adult nearby.", "skyblue"),
    ("garage", "A mechanic repairs a motorcycle inside a garage.", "orange"),
    ("cat", "A cat lies beside an open laptop.", "seagreen"),
]

metadata = []
for index, (name, caption, color) in enumerate(examples):
    image = Image.new("RGB", (640, 360), color=color)
    draw = ImageDraw.Draw(image)
    draw.text((30, 150), caption, fill="white")
    frame_path = FRAMES_DIR / f"{name}.jpg"
    image.save(frame_path)
    metadata.append({
        "id": name,
        "transcript": caption,
        "frame_path": str(frame_path),
        "video_path": f"demo_video_{index // 2 + 1}.mp4",
        "timestamp_ms": index * 10_000,
    })

metadata

## Add neighboring transcript context

Short captions can be ambiguous. The article joins neighboring transcript segments before embedding; this helper implements the same idea with an explicit, symmetric window.

In [ ]:
def add_neighbor_context(records: list[dict], radius: int = 1) -> list[dict]:
    transcripts = [record["transcript"] for record in records]
    enriched = []
    for index, record in enumerate(records):
        start = max(0, index - radius)
        stop = min(len(records), index + radius + 1)
        neighbor_text = " ".join(transcripts[start:stop])
        enriched.append({
            **record,
            # Repeat the focal caption so context enriches rather than dominates it.
            "embedding_text": (
                f"{record['transcript']} {record['transcript']} "
                f"Neighbor context: {neighbor_text}"
            ),
        })
    return enriched

metadata = add_neighbor_context(metadata, radius=1)
metadata[1]

## Create multimodal embeddings with CLIP

CLIP maps images and text into a shared space. We normalize the image and caption vectors, average them for indexing, and use a text vector for retrieval.

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
print(f"Using {DEVICE}")

In [ ]:
def normalize(vector: torch.Tensor) -> torch.Tensor:
    return vector / vector.norm(dim=-1, keepdim=True)

@torch.inference_mode()
def text_embedding(text: str) -> np.ndarray:
    inputs = processor(text=[text], return_tensors="pt", padding=True)
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    return normalize(model.get_text_features(**inputs))[0].cpu().numpy()

@torch.inference_mode()
def image_embedding(image: Image.Image) -> np.ndarray:
    inputs = processor(images=image, return_tensors="pt")
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    return normalize(model.get_image_features(**inputs))[0].cpu().numpy()

def joint_embedding(image_path: str, text: str) -> np.ndarray:
    image = Image.open(image_path).convert("RGB")
    # Keep text queries aligned with the caption while adding visual evidence.
    combined = 0.25 * image_embedding(image) + 0.75 * text_embedding(text)
    return combined / np.linalg.norm(combined)

## Index the records in LanceDB

In [ ]:
rows = []
for record in metadata:
    vector = joint_embedding(record["frame_path"], record["embedding_text"])
    rows.append({**record, "vector": vector.astype("float32").tolist()})

database = lancedb.connect(DATA_DIR / "lancedb")
table = database.create_table("video_frames", data=rows, mode="overwrite")
table.to_pandas()[["id", "transcript", "frame_path"]]

## Retrieve ranked multimodal evidence

In [ ]:
def retrieve(query: str, top_k: int = 3) -> pd.DataFrame:
    query_vector = text_embedding(query).astype("float32")
    return (
        table.search(query_vector)
        .limit(top_k)
        .to_pandas()[["id", "transcript", "frame_path", "_distance"]]
    )

results = retrieve("A person working near a spacecraft in orbit", top_k=3)
results

In [ ]:
for rank, result in results.iterrows():
    print(f"{rank + 1}. {result['transcript']} (distance={result['_distance']:.4f})")
    display(Image.open(result["frame_path"]))

## Save the indexed metadata

The LanceDB table persists locally. Saving the non-vector metadata separately also makes the pipeline easier to inspect and reproduce.

In [ ]:
METADATA_PATH = DATA_DIR / "indexed_metadata.json"
METADATA_PATH.write_text(json.dumps(metadata, indent=2))
print(f"Saved {len(metadata)} records to {METADATA_PATH}")

## Next step

The retriever now returns grounded text and visual evidence. Part 6 will pass that evidence to a large vision-language model for answer generation.